In [3]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

In [4]:
model_df = pd.read_csv('../data/processed/nhl_model_data.csv')

In [5]:
model_df['home_team_win'] = model_df['home_goals_for'] > model_df['away_goals_for']
model_df['home_team_win'] = model_df['home_team_win'].astype(int)

In [6]:
feature_cols = [
    col for col in model_df.columns 
    if (
        col.startswith("home_rolling_") or col.startswith("away_rolling_")
    )
    and (
        col.endswith("_avg5") or col.endswith("_avg10")
    )   
]      

In [7]:
print(len(feature_cols))

48


In [8]:
model_df = model_df.dropna(subset=feature_cols).reset_index(drop=True)

X = model_df[feature_cols]
y = model_df['home_team_win']

print(X.shape)
print(y.shape)

(1292, 48)
(1292,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)

preds = log_reg_pipeline.predict(X_test)
probs = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, preds))
print("ROC AUC:", roc_auc_score(y_test, probs))
print("Classification Report:\n", classification_report(y_test, preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

Accuracy: 0.5521235521235521
ROC AUC: 0.5665703623450102
Classification Report:
               precision    recall  f1-score   support

           0       0.59      0.60      0.59       142
           1       0.50      0.50      0.50       117

    accuracy                           0.55       259
   macro avg       0.55      0.55      0.55       259
weighted avg       0.55      0.55      0.55       259

Confusion Matrix:
 [[85 57]
 [59 58]]


/Users/mnpol/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/mnpol/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/mnpol/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/mnpol/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/mnpol/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_s

In [10]:
import numpy as np
from sklearn.metrics import accuracy_score

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": log_reg_pipeline.named_steps["model"].coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df.sort_values("abs_coefficient", ascending=False).head(25)

,feature,coefficient,abs_coefficient
16,home_rolling_faceoff_win_pct_avg10,-0.367822,0.367822
11,home_rolling_pims_avg5,-0.286530,0.286530
4,home_rolling_faceoff_win_pct_avg5,0.283776,0.283776
32,away_rolling_blocked_shots_avg5,-0.231249,0.231249
23,home_rolling_pims_avg10,0.230926,0.230926
27,away_rolling_shots_against_avg5,0.225578,0.225578
14,home_rolling_shots_for_avg10,0.197011,0.197011
19,home_rolling_hits_avg10,0.196529,0.196529
8,home_rolling_blocked_shots_avg5,-0.182937,0.182937
7,home_rolling_hits_avg5,-0.181280,0.181280
